In [1]:
import os, pathlib
import pandas as pd
import numpy as np

In [2]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"
laws_de_path = "../data/laws_de.csv"
val_path = "../data/val.csv"
court_consdr = "../data/court_considerations.csv"

In [3]:
train = pd.read_csv(train_path)
val = pd.read_csv(val_path)
law = pd.read_csv(laws_de_path)
court = pd.read_csv(court_consdr)

In [4]:
train.shape

(1139, 3)

In [5]:
law.shape

(175933, 3)

In [6]:
# Get all unique citation from training set
all_train_citation = set()
for citation in train["gold_citations"]  :
    all_train_citation.update(citation.split(";"))

print(f"total unique citation in train citation {all_train_citation}")
print(f"total len of unique citation in all_train_citation = {len(all_train_citation)}")

# check how many of these in law citations
law_citation = set(law["citation"].unique())
overlap_law = all_train_citation.intersection(law_citation)

#check how many of these in court citations
court_citation = set(court["citation"].unique())
overlap_court = all_train_citation.intersection(court_citation)

print(f"overlap with law citation = {overlap_law}")
print(f"len of the overlap citation = {len(overlap_law)}")

print(f"overlap with court citation = {overlap_court}")
print(f"len of the overlap citation = {len(overlap_court)}")

total unique citation in train citation {'Art. 651 Abs. 1 ZGB', 'Art. 31 Abs. 1 GSchG', 'Art. 204 Abs. 1 ZGB', 'Art. 18 BankG', 'Art. 18b DBG', 'Art. 36 BV', 'Art. 236 ZPO', 'Art. 94 Abs. 1 GBV', 'Art. 77 Abs. 1 OR', 'Art. 86a ZGB', 'Art. 49 Abs. 1 BV', 'Art. 155 IPRG', 'Art. 285 Abs. 2 ZGB', 'Art. 14 Abs. 2 FIDLEG', 'Art. 216 Abs. 1 ZGB', 'Art. 78 Abs. 2 BV', 'Art. 93 Abs. 2 BV', 'Art. 42 BV', 'Art. 36 Abs. 1 FIDLEG', 'Art. 13 Abs. 1 BV', 'Art. 169 ZPO', 'Art. 530 Abs. 1 OR', 'Art. 29 Abs. 5 BEG', 'Art. 824 Abs. 2 ZGB', 'Art. 366 ZGB', 'Art. 46a VwVG', 'Art. 20 IPRG', 'Art. 216 Abs. 2 ZGB', 'Art. 40 Abs. 3 RPV', 'Art. 98 Abs. 3 GBV', 'Art. 482 ZGB', 'Art. 24b Abs. 2 RPG', 'Art. 9 Abs. 1 KGTG', 'Art. 181 IPRG', 'Art. 11 Abs. 1 UVG', 'Art. 74a Abs. 2 IRSG', 'Art. 9 Abs. 2 UVG', 'Art. 36 FIDLEG', 'Art. 80 Abs. 4 GBV', 'Art. 145 Abs. 1 ZPO', 'Art. 45a Abs. 3 IPRG', 'Art. 224 Abs. 1 ZPO', 'Art. 9 RAG', 'Art. 81 Abs. 2 BVG', 'Art. 349a Abs. 2 OR', 'Art. 7 Abs. 2 RAG', 'Art. 126 FinfraV', 'A

In [7]:
print(len(all_train_citation))

2695


In [8]:
#print(f"Missing citation = {len(all_train_citation) - len(overlap)}")

In [9]:
# Check whether we need chunking strategies or not
law["text_word_count"] = law["text"].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)
print(law["text_word_count"].describe())

count    175933.000000
mean         31.924392
std          37.209511
min           1.000000
25%          16.000000
50%          24.000000
75%          37.000000
max        4505.000000
Name: text_word_count, dtype: float64


In [10]:
law.head()

,citation,text,title,text_word_count
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,259
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,35
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,143
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,65
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,27


In [11]:
val.head()

,query_id,query,gold_citations
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
2,val_003,"A. Rivera, a Peruvian national born in 1994 an...",Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...
3,val_004,"Mr. Dalton, born in 1941 and resident in a sma...",Art. 505 Abs. 1 ZGB;Art. 467 ZGB;Art. 469 Abs....
4,val_005,"A parent, separated from their co-parent since...",Art. 133 Abs. 1 ZGB;Art. 133 Abs. 2 ZGB;Art. 2...


In [12]:
#r"Art\.\s*\d+(?:\s+(?:Abs\.|lit\.|Ziff\.)\s+[a-z0-9]+)*\s+[A-Za-z]+"

In [13]:
train["query"][6]

'Erblasserin E hinterlässt zwei Kinder, Ki und K2, sowie ein Nettovermögen von CHF 500000. Es gilt die gesetzliche Erbfolge. Zu Lebzeiten hat K1 von E CHF 100°000 als Unterstützung zur Gründung seines Unternehmens geschenkt bekommen: K2 hat von E ebenfalls CHF 100°000 ohne Gegenleis- tung erhalten, um sich für sein Freizeitvergnügen einen Oldtimer zu kaufen. E hat K2 zudem aus- drücklich von etwaigen Ausgleichungspflichten befreit. \nSind K1 und K2 zur Ausgleichung verpflichtet? Prüfen Sie alle Voraussetzungen der Ausglei- chungspflicht. Ausführungen zur konkreten Höhe der Ausgleichungspflicht sind nicht erforderlich.'

In [14]:
# Validations query contain the citations
val["query"][2]

'A. Rivera, a Peruvian national born in 1994 and with no prior convictions in the forum state, is accused of having, between 5 March and 9 March 2024, together with three accomplices (B. L., C. M. and D. S.) taken part in a series of offenses including the theft and driving off of delivery vans, forcible entry into a riverside storage unit and the theft of items (notably a replica handgun), the assault of a witness with a metal bar, the discharge of a shot toward the river, and on 9 March 2024 the theft of a further van followed by a high‑speed pursuit in which shots were fired toward law‑enforcement officers; A. Rivera was located in a neighbouring state on 18 March 2024, returned to the prosecuting country on 6 June 2024, interviewed and admitted being present but denied active participation. The authority responsible for ordering coercive measures imposed pretrial detention on several occasions and extended it until 1 December 2024, citing flight and collusion risks based on video f

In [15]:
import re
# Extract the citation from the validation query
pattern = r"Art\.\s*\d+(?:\s+(?:Abs\.|lit\.|Ziff\.)\s+[a-z0-9]+)*\s+[A-Za-z]+"

# validation query
val_query = val["query"][0]

print("Testing the Regex on the validation query.....")
extracted_citations = re.findall(pattern, val_query)
print(f"extracted_citations: {extracted_citations}")

print("validation golder citations")
val["gold_citations"][0]

print("search on the law_de text whether it has val query citation or not")
for cite in extracted_citations :
    found = law[law["citation"].str.contains(cite, regex=False, na=False)]
    print(f"The matching string are: {found}")
    print(f"len of total matching value {len(found)}")

Testing the Regex on the validation query.....
extracted_citations: ['Art. 221 Abs. 1 lit. b StPO']
validation golder citations
search on the law_de text whether it has val query citation or not
The matching string are: Empty DataFrame
Columns: [citation, text, title, text_word_count]
Index: []
len of total matching value 0


In [16]:
#law[law["citation"].str.contains(cite, regex=False, na=False)]

validation query contains multiples citations. each row can have contain more than one citations.  
But law_de has don't exactly contain val_query citation. it contain some modification verstion of  
citations.  
We have to focus on this matter. Because base on query we have to extract the citation from the  
law_de and court_considerations. 

In [17]:
court.sample(5)

,citation,text
438754,8C_282/2022 E. 1,Das Bundesgericht prüft die Eintretensvorausse...
2110806,8C_846/2011 19.04.2012 E. 2,2.1 Nach Art. 90 BGG ist die Beschwerde zuläss...
1820176,5C.45/2006 15.03.2006 E. C,Gegen diesen Entscheid hat der Berufungskläger...
380929,1C_452/2015 E. 3,Les recourants verseront aux intimés la somme ...
892435,4A_74/2015 E. 3,Le recourant versera à l'intimée une indemnité...


In [18]:
court.sample()["text"][0:1]

1889475    Dieses Urteil wird dem Beschwerdeführer, der S...
Name: text, dtype: object

### Court Considerations

In [19]:
# check the shape of the dataset
print(f"Shape of the dataset: {court.shape}")

# check whether it is required chunking or not
court["word_count"] = court["text"].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

print("Court text word count statistics")
display(court["word_count"].describe())

Shape of the dataset: (2476315, 2)
Court text word count statistics


count    2.476315e+06
mean     1.374033e+02
std      1.994214e+02
min      1.000000e+00
25%      2.700000e+01
50%      7.100000e+01
75%      1.720000e+02
max      1.177100e+04
Name: word_count, dtype: float64

In [20]:
court.head()

,citation,text,word_count
0,BGE 139 I 2 E. 1.12.2011,betr. Verweigerung der Beiladung seien gutzuhe...,12
1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...,112
2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents...",65
3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...,39
4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...,42


In [21]:
2.476315e+06

2476315.0

In [22]:
# Check for a specific pattern (e.g., 'BGE') in this small chunk
bge_count = court['citation'].str.contains('BGE', na=False).sum()
print(f"\nNumber of 'BGE' (Leading Decisions) in this chunk: {bge_count}")


Number of 'BGE' (Leading Decisions) in this chunk: 96465


In [23]:
val["query"][0]

'May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of proportionality when the accused—detained after an alleged late‑night assault and theft of a courier satchel containing, inter alia, €5,600—was remanded by an order dated 18 October 2024 for a maximum period up to 15 January 2025, the prosecutor sought an extension on 10 December 2024 primarily citing a concrete risk that the detainee would influence witnesses or tamper with evidence and a risk of reoffending, while the detainee opposed the extension on the ground that most witnesses have already been interviewed, the investigative steps still pending are essentially technical (phone data extraction, CCTV image analysis, and bank record checks), his release would therefore not jeopardize the inquiry, and the alleged victim has withdrawn the complaint—i.e. does the asserted concrete risk of collusion and considerations of propo

In [24]:
gold_target = "Art. 221 Abs. 1 StPO"

In [25]:
match_ = law[law['citation'] == gold_target]

In [26]:
match_.iloc[0]["citation"]

'Art. 221 Abs. 1 StPO'

In [27]:
val.iloc[0]

query_id                                                    val_001
query             May a court lawfully order a three‑month exten...
gold_citations    Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
Name: 0, dtype: object

In [28]:
val.iloc[1]

query_id                                                    val_002
query             A claimant holding a national vocational diplo...
gold_citations    Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
Name: 1, dtype: object

In [29]:
stpo = law[law["citation"].str.contains("StPO", na=False, regex=False)]

In [30]:
len(stpo)

1441

In [31]:
stpo[["citation", "text"]].sample(5)

,citation,text
131505,Art. 202 Abs. 3 StPO,3 Bei der Festlegung des Zeitpunkts wird auf d...
131780,Art. 289 Abs. 1 StPO,1 Der Einsatz einer verdeckten Ermittlerin ode...
131245,Art. 106 Abs. 1 StPO,1 Die Partei kann Verfahrenshandlungen nur gül...
131470,Art. 186 Abs. 1 StPO,1 Staatsanwaltschaft und Gerichte können eine ...
131500,Art. 200 StPO,Zur Durchsetzung von Zwangsmassnahmen darf als...


In [32]:
val["query"][1]

'A claimant holding a national vocational diploma in warehouse operations worked intermittently as a storage technician from 10 March to 20 September 2022 and was entered as job-seeking on 1 October 2022. From mid-2021 onwards he has suffered from a chronic allergic respiratory disorder (eosinophilic allergic asthma with nasal/ocular symptoms) with raised IgE to seasonal pollens and household mites; he required inpatient treatment for an acute exacerbation in August 2022. Allergy specialists advised placement in a temperature-controlled, low-dust workplace, yet an occupational pulmonologist’s assessment dated 15 April 2023 concluded the claimant retained full capacity for employment and that the condition was stable. Subsequent brief medical notes from his family doctor (dated 12 February and 3 March 2024) state that he is completely unable to perform his usual duties as a storage technician but could carry out adapted work that excludes exposure to dust, pollens and moulds; in early 2

In [33]:
val_citation = "Art. 17 LAI"

In [34]:
stpo = law[law["citation"].str.contains("LAI", na=False, regex=False)]

In [35]:
stpo

,citation,text,title,text_word_count


In [36]:
lai = court[court["citation"].str.contains("LAI", na=False, regex=False)]

In [37]:
lai

,citation,text,word_count


In [38]:
text = val["query"][0]

pattern = r"Art\.\s*\d+(?:\s+(?:Abs\.|lit\.|Ziff\.)\s+[a-z0-9]+)*\s+[A-Za-z]+"
raw_matches = re.findall(pattern, text)

In [39]:
raw_matches

['Art. 221 Abs. 1 lit. b StPO']

In [40]:
#text = val["query"].apply(lambda x : re.findall(pattern, x))

In [41]:

clean_cit = re.sub(r'\s+lit\.\s+[a-z]+', '', raw_matches[0])
clean_cit = re.sub(r'\s+Ziff\.\s+\d+', '', clean_cit)    

In [42]:
clean_cit

'Art. 221 Abs. 1 StPO'

In [43]:
clean_cit.split()

['Art.', '221', 'Abs.', '1', 'StPO']

In [44]:
" ".join(clean_cit.split())

'Art. 221 Abs. 1 StPO'

In [45]:
text = train["query"].apply(lambda x : re.findall(pattern, x))

In [46]:
text

0       []
1       []
2       []
3       []
4       []
        ..
1134    []
1135    []
1136    []
1137    []
1138    []
Name: query, Length: 1139, dtype: object

In [47]:
["string"][0]

'string'

In [48]:
val.sample()

,query_id,query,gold_citations
6,val_007,An heirship claims title to a vintage pocket c...,Art. 98 Abs. 2 IPRG;Art. 100 Abs. 1 IPRG;Art. ...


In [49]:
val["query"].tolist()

['May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of proportionality when the accused—detained after an alleged late‑night assault and theft of a courier satchel containing, inter alia, €5,600—was remanded by an order dated 18 October 2024 for a maximum period up to 15 January 2025, the prosecutor sought an extension on 10 December 2024 primarily citing a concrete risk that the detainee would influence witnesses or tamper with evidence and a risk of reoffending, while the detainee opposed the extension on the ground that most witnesses have already been interviewed, the investigative steps still pending are essentially technical (phone data extraction, CCTV image analysis, and bank record checks), his release would therefore not jeopardize the inquiry, and the alleged victim has withdrawn the complaint—i.e. does the asserted concrete risk of collusion and considerations of prop

In [50]:
val.shape

(10, 3)

In [74]:
def split_by_articles(text: str) -> list[str]:
    pattern = r"(Art\.?\s*\d+[a-zA-Z]*)"
    parts = re.split(pattern, text)
    print(parts)
    print(f"len = {len(parts)}")
    chunks = []
    for i in range(1, len(parts), 2):
        print("start for loop...")
        title = parts[i]
        print(f"tilte: {title}")
        content = parts[i+1] if i+1 < len(parts) else ""
        print(f"content: {content}")
        chunks.append(title + " " + content)

    return chunks if chunks else [text]

In [79]:
split_by_articles(law["text"].loc[84387])

['2 Ins Handelsregister müssen eingetragen werden:a. das Datum der Änderung der Statuten;\nb. die Höhe und die Währung des Aktienkapitals und der darauf geleisteten Einlagen sowie Anzahl, Nennwert und Art der Aktien.']
len = 1


['2 Ins Handelsregister müssen eingetragen werden:a. das Datum der Änderung der Statuten;\nb. die Höhe und die Währung des Aktienkapitals und der darauf geleisteten Einlagen sowie Anzahl, Nennwert und Art der Aktien.']

In [56]:
law["text"].loc[0]

'Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäude des Bundesrathauses im roten Quartier der Stadt Bern, mit Nr. 229 bezeichnet, nebst den in demselben enthaltenen Einrichtungen und Mobilien, welche der Einwohnergemeinde angehören, und unter Vorbehalt der im Artikel 62 von der Einwohnergemeinde reservierten Einrichtungen und Gegenstände;\nb. den zwischen den Seitenflügeln des Bundesrathauses und nördlich von dem Mittelbau desselben befindlichen innern Hof von ungefähr 25 000 Quadratfuss Oberfläche.\n Derselbe wird abgetreten bis zu einer in Verlängerung der Nordfassaden der Seitenflügel gezogenen Linie.\n Der in diesem Hofe befindliche Brunnen verbleibt der Einwohnergemeinde, welche denselben in gutem Zustande erhalten und ohne Genehmigung des Bundesrates an dem jetzigen baulichen Zustand mit Inbegriff der Statuen keine Veränderung vornehmen soll.\n Sie wird den Brunnen wie bis anhin mit Wasser versehen.\n Die Eidgenosse

In [1]:
text = "Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum"
word = text.split()

In [2]:
word

['Die',
 'Einwohnergemeinde',
 'Bern',
 'tritt',
 'der',
 'Schweizerischen',
 'Eidgenossenschaft',
 'unentgeltlich',
 'als',
 'Eigentum']

In [3]:
chunk = " ".join(word)

In [4]:
chunk

'Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum'

In [15]:
def split_by_paragraph(text: str):
    paragraphs = text.split("\n")
    return [p.strip() for p in paragraphs if p.strip()]

In [ ]:
# import re
# from typing import List

# # Step 1: split by legal markers
# def split_by_articles(text: str) -> List[str]:
#     pattern = r"(Art\.?\s*\d+[a-zA-Z]*)"
#     parts = re.split(pattern, text)

#     chunks = []
#     for i in range(1, len(parts), 2):
#         title = parts[i]
#         content = parts[i+1] if i+1 < len(parts) else ""
#         chunks.append(title + " " + content)

#     return chunks if chunks else [text]

In [17]:
def token_chunk(text, chunk_size=250, overlap=50):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        start = end - overlap

    return chunks

In [18]:
def hybrid_chunk(text):
    # Try article split
    chunks = split_by_articles(text)

    final_chunks = []

    for chunk in chunks:
        if len(chunk.split()) < 300:
            final_chunks.append(chunk)
        else:
            # fallback to paragraph
            paras = split_by_paragraph(chunk)

            for p in paras:
                if len(p.split()) < 300:
                    final_chunks.append(p)
                else:
                    # final fallback
                    final_chunks.extend(token_chunk(p))

    return final_chunks

In [19]:
def build_chunks(laws_df):
    all_chunks = []

    for _, row in laws_df.iterrows():
        citation = row["citation"]
        title = row.get("title", "")
        text = row["text"]

        chunks = hybrid_chunk(text)

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "citation": citation,
                "title": title,
                "chunk_id": f"{citation}_{i}",
                "text": chunk
            })

    return all_chunks

In [20]:
build_chunks(laws_df=law)

[{'citation': 'Art. 1 112',
  'title': 'Übereinkunft vom 22. Juni 1875 zwischen dem Schweizerischen Bundesrate und dem Einwohnergemeinderate der Stadt Bern betreffend die Leistungen der Stadt Bern an den Bundessitz',
  'chunk_id': 'Art. 1 112_0',
  'text': 'Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäude des Bundesrathauses im roten Quartier der Stadt Bern, mit Nr. 229 bezeichnet, nebst den in demselben enthaltenen Einrichtungen und Mobilien, welche der Einwohnergemeinde angehören, und unter Vorbehalt der im Artikel 62 von der Einwohnergemeinde reservierten Einrichtungen und Gegenstände;\nb. den zwischen den Seitenflügeln des Bundesrathauses und nördlich von dem Mittelbau desselben befindlichen innern Hof von ungefähr 25 000 Quadratfuss Oberfläche.\n Derselbe wird abgetreten bis zu einer in Verlängerung der Nordfassaden der Seitenflügel gezogenen Linie.\n Der in diesem Hofe befindliche Brunnen verbleibt der Einwohnerge

In [7]:
for i, j in law[0:5].iterrows() :
    print(f"i = {i}")
    #print(f"j = {j}")
    print(j)
    print(f"type = {type(j)}")

i = 0
citation                                           Art. 1 112
text        Die Einwohnergemeinde Bern tritt der Schweizer...
title       Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
Name: 0, dtype: object
type = <class 'pandas.core.series.Series'>
i = 1
citation                                           Art. 2 112
text        Die Einwohnergemeinde Bern wird ferner der Sch...
title       Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
Name: 1, dtype: object
type = <class 'pandas.core.series.Series'>
i = 2
citation                                    Art. 3 Abs. 1 112
text        1 Falls die Schweizerische Eidgenossenschaft z...
title       Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
Name: 2, dtype: object
type = <class 'pandas.core.series.Series'>
i = 3
citation                                    Art. 3 Abs. 2 112
text        2 Durch Anlage des neuen Verwaltungsgebäudes a...
title       Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
Name: 3, dtype: object
type = <cla

In [25]:
sample = law.iloc[0]["text"]
chunks = hybrid_chunk(sample)

for c in chunks[0:20]:
    print(c)
    print("----")

Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäude des Bundesrathauses im roten Quartier der Stadt Bern, mit Nr. 229 bezeichnet, nebst den in demselben enthaltenen Einrichtungen und Mobilien, welche der Einwohnergemeinde angehören, und unter Vorbehalt der im Artikel 62 von der Einwohnergemeinde reservierten Einrichtungen und Gegenstände;
b. den zwischen den Seitenflügeln des Bundesrathauses und nördlich von dem Mittelbau desselben befindlichen innern Hof von ungefähr 25 000 Quadratfuss Oberfläche.
 Derselbe wird abgetreten bis zu einer in Verlängerung der Nordfassaden der Seitenflügel gezogenen Linie.
 Der in diesem Hofe befindliche Brunnen verbleibt der Einwohnergemeinde, welche denselben in gutem Zustande erhalten und ohne Genehmigung des Bundesrates an dem jetzigen baulichen Zustand mit Inbegriff der Statuen keine Veränderung vornehmen soll.
 Sie wird den Brunnen wie bis anhin mit Wasser versehen.
 Die Eidgenossenschaf

In [12]:
law["text"][3]

'2 Durch Anlage des neuen Verwaltungsgebäudes an hier bezeichneter Stelle übernimmt die Eidgenossenschaft bezüglich der Erstellung der Trottoirs und Trottoirsrinnen längs den Strassen, welche an das von ihr erworbene Grundeigentum grenzen, die gleichen Verpflichtungen, welche durch Artikel 5 der Übereinkunft vom 29. Januar 1872 zwischen Staat und Gemeinde Bern den Käufern von Bauparzellen auf dem Territorium des nördlichen Abschnittes der kleinen Schanze überbunden worden sind.'

In [17]:
import re

In [ ]:
def split_by_articles(text: str) -> list[dict[str, str]]:
    """
    Split legal text by articles. Handles formats like:
    - Art. 1, Art. 1a, Art. 1 Abs. 1, Abs. 2, § 1, etc.
    Returns list of dicts with article info and content
    """
    # Enhanced pattern for German/Swiss legal format
    article_pattern = r"((?:Art\.?|§)\s*\d+[a-zA-Z]*(?:\s+Abs\.?\s*\d+)?)"
    parts = re.split(article_pattern, text)

    chunks = []
    for i in range(1, len(parts), 2):
        article_header = parts[i].strip()
        content = parts[i+1].strip() if i+1 < len(parts) else ""
        
        if content:  # Only include if there's content
            chunks.append({
                "article": article_header,
                "content": content,
                "full_text": article_header + " " + content
            })
    print(chunks)
    return chunks if chunks else [{"article": "Full Text", "content": text, "full_text": text}]

In [19]:
for _, row in law.iterrows() :
    text = row["text"]
    split_by_articles(text)

In [24]:
# Get chunks from a sample law text
sample_text = law.iloc[0]["text"]
chunks = split_by_articles(sample_text)

# Display the chunks
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:")
    print(chunk)
    print("----")

Chunk 0:
{'article': 'Full Text', 'content': 'Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäude des Bundesrathauses im roten Quartier der Stadt Bern, mit Nr. 229 bezeichnet, nebst den in demselben enthaltenen Einrichtungen und Mobilien, welche der Einwohnergemeinde angehören, und unter Vorbehalt der im Artikel 62 von der Einwohnergemeinde reservierten Einrichtungen und Gegenstände;\nb. den zwischen den Seitenflügeln des Bundesrathauses und nördlich von dem Mittelbau desselben befindlichen innern Hof von ungefähr 25 000 Quadratfuss Oberfläche.\n Derselbe wird abgetreten bis zu einer in Verlängerung der Nordfassaden der Seitenflügel gezogenen Linie.\n Der in diesem Hofe befindliche Brunnen verbleibt der Einwohnergemeinde, welche denselben in gutem Zustande erhalten und ohne Genehmigung des Bundesrates an dem jetzigen baulichen Zustand mit Inbegriff der Statuen keine Veränderung vornehmen soll.\n Sie wird den Brunnen wie bi

In [5]:
test_query = val['query'].iloc[0]

In [6]:
test_query

'May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of proportionality when the accused—detained after an alleged late‑night assault and theft of a courier satchel containing, inter alia, €5,600—was remanded by an order dated 18 October 2024 for a maximum period up to 15 January 2025, the prosecutor sought an extension on 10 December 2024 primarily citing a concrete risk that the detainee would influence witnesses or tamper with evidence and a risk of reoffending, while the detainee opposed the extension on the ground that most witnesses have already been interviewed, the investigative steps still pending are essentially technical (phone data extraction, CCTV image analysis, and bank record checks), his release would therefore not jeopardize the inquiry, and the alleged victim has withdrawn the complaint—i.e. does the asserted concrete risk of collusion and considerations of propo

In [10]:
val["query"][0]

'May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of proportionality when the accused—detained after an alleged late‑night assault and theft of a courier satchel containing, inter alia, €5,600—was remanded by an order dated 18 October 2024 for a maximum period up to 15 January 2025, the prosecutor sought an extension on 10 December 2024 primarily citing a concrete risk that the detainee would influence witnesses or tamper with evidence and a risk of reoffending, while the detainee opposed the extension on the ground that most witnesses have already been interviewed, the investigative steps still pending are essentially technical (phone data extraction, CCTV image analysis, and bank record checks), his release would therefore not jeopardize the inquiry, and the alleged victim has withdrawn the complaint—i.e. does the asserted concrete risk of collusion and considerations of propo

In [14]:
law["text"][5:8]

5    2 Sie übernimmt auch die Verpflichtung, die er...
6    3 Im Fall die Schweizerische Eidgenossenschaft...
7    1 Sollte infolge förmlichen Beschlusses der ko...
Name: text, dtype: object

In [15]:
law["text"][6]

'3 Im Fall die Schweizerische Eidgenossenschaft von der ihr durch Artikel 3 eingeräumten Befugnis zur Beanspruchung von Land auf dem frühern Territorium der kleinen Schanze Gebrauch machen würde, so übernimmt überdies die Einwohnergemeinde auch dem Bunde gegenüber die Verpflichtung, die südlich von dem neu erstellten Verwaltungsgebäude verbleibenden Teile der kleinen Schanze als öffentliche Promenadenanlage zu erstellen und zu unterhalten.'

In [17]:
[str(doc).lower().split() for doc in law['text'][5:8].fillna("")]

[['2',
  'sie',
  'übernimmt',
  'auch',
  'die',
  'verpflichtung,',
  'die',
  'erwähnte',
  'terrasse',
  'zwischen',
  'dem',
  'bundesrathause',
  'und',
  'der',
  'vannazhalde',
  'als',
  'öffentliche',
  'anlage',
  'zu',
  'erhalten.'],
 ['3',
  'im',
  'fall',
  'die',
  'schweizerische',
  'eidgenossenschaft',
  'von',
  'der',
  'ihr',
  'durch',
  'artikel',
  '3',
  'eingeräumten',
  'befugnis',
  'zur',
  'beanspruchung',
  'von',
  'land',
  'auf',
  'dem',
  'frühern',
  'territorium',
  'der',
  'kleinen',
  'schanze',
  'gebrauch',
  'machen',
  'würde,',
  'so',
  'übernimmt',
  'überdies',
  'die',
  'einwohnergemeinde',
  'auch',
  'dem',
  'bunde',
  'gegenüber',
  'die',
  'verpflichtung,',
  'die',
  'südlich',
  'von',
  'dem',
  'neu',
  'erstellten',
  'verwaltungsgebäude',
  'verbleibenden',
  'teile',
  'der',
  'kleinen',
  'schanze',
  'als',
  'öffentliche',
  'promenadenanlage',
  'zu',
  'erstellen',
  'und',
  'zu',
  'unterhalten.'],
 ['1',
  'soll

In [17]:
from rank_bm25 import BM25Okapi
import time

print("Preparing BM25 on the FULL laws corpus...")
start_time = time.time()

# Simple tokenization: lowercasing and splitting by space 
# We fill NaN values with empty string to avoid errors
tokenized_corpus = [str(doc).lower().split() for doc in law['text'].fillna("")]

# Initialize BM25 model
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 indexing finished in {time.time() - start_time:.2f} seconds.")

# Prepare the same test query 
test_query = val['query'].iloc[0]
tokenized_query = test_query.lower().split()

print("\nSearching with BM25...")
# Get top 3 scores and their indices 
bm25_scores = bm25.get_scores(tokenized_query)
top_k = 3

Preparing BM25 on the FULL laws corpus...
BM25 indexing finished in 3.76 seconds.

Searching with BM25...


In [18]:
bm25_scores

array([2.60583334, 0.96223219, 5.3279724 , ..., 0.59116989, 0.        ,
       1.48323831], shape=(175933,))

In [19]:
np.argsort(bm25_scores)

array([175894,  87281, 175843, ...,  62462,  50367,   7249],
      shape=(175933,))

In [20]:
np.argsort(bm25_scores)[-3:]# [::-1]

array([62462, 50367,  7249])

In [30]:
val[val['gold_citations'] == "Art. 3 Abs. 1 221.434"]

,query_id,query,gold_citations


In [29]:
law["citation"][7249]

'Art. 3 Abs. 1 221.434'

In [21]:
top_indices = np.argsort(bm25_scores)[-top_k:][::-1]

In [11]:
top_indices

array([21, 22,  3])

In [15]:
tokenized_corpus = [str(doc).lower().split() for doc in law['text'][0:50].fillna("")]

In [16]:
tokenized_corpus

[['die',
  'einwohnergemeinde',
  'bern',
  'tritt',
  'der',
  'schweizerischen',
  'eidgenossenschaft',
  'unentgeltlich',
  'als',
  'eigentum',
  'ab:a.',
  'das',
  'gebäude',
  'des',
  'bundesrathauses',
  'im',
  'roten',
  'quartier',
  'der',
  'stadt',
  'bern,',
  'mit',
  'nr.',
  '229',
  'bezeichnet,',
  'nebst',
  'den',
  'in',
  'demselben',
  'enthaltenen',
  'einrichtungen',
  'und',
  'mobilien,',
  'welche',
  'der',
  'einwohnergemeinde',
  'angehören,',
  'und',
  'unter',
  'vorbehalt',
  'der',
  'im',
  'artikel',
  '62',
  'von',
  'der',
  'einwohnergemeinde',
  'reservierten',
  'einrichtungen',
  'und',
  'gegenstände;',
  'b.',
  'den',
  'zwischen',
  'den',
  'seitenflügeln',
  'des',
  'bundesrathauses',
  'und',
  'nördlich',
  'von',
  'dem',
  'mittelbau',
  'desselben',
  'befindlichen',
  'innern',
  'hof',
  'von',
  'ungefähr',
  '25',
  '000',
  'quadratfuss',
  'oberfläche.',
  'derselbe',
  'wird',
  'abgetreten',
  'bis',
  'zu',
  'einer',

In [8]:
law['text'][0:5].to_list()

['Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäude des Bundesrathauses im roten Quartier der Stadt Bern, mit Nr. 229 bezeichnet, nebst den in demselben enthaltenen Einrichtungen und Mobilien, welche der Einwohnergemeinde angehören, und unter Vorbehalt der im Artikel 62 von der Einwohnergemeinde reservierten Einrichtungen und Gegenstände;\nb. den zwischen den Seitenflügeln des Bundesrathauses und nördlich von dem Mittelbau desselben befindlichen innern Hof von ungefähr 25 000 Quadratfuss Oberfläche.\n Derselbe wird abgetreten bis zu einer in Verlängerung der Nordfassaden der Seitenflügel gezogenen Linie.\n Der in diesem Hofe befindliche Brunnen verbleibt der Einwohnergemeinde, welche denselben in gutem Zustande erhalten und ohne Genehmigung des Bundesrates an dem jetzigen baulichen Zustand mit Inbegriff der Statuen keine Veränderung vornehmen soll.\n Sie wird den Brunnen wie bis anhin mit Wasser versehen.\n Die Eidgenoss

In [5]:
val.size

30

In [9]:
size = 7
for i in range(0, len(val), size) :
    chunk = val.iloc[i:i+size]
    print(f"i = {i} \n{chunk}, \n")

i = 0 
  query_id                                              query  \
0  val_001  May a court lawfully order a three‑month exten...   
1  val_002  A claimant holding a national vocational diplo...   
2  val_003  A. Rivera, a Peruvian national born in 1994 an...   
3  val_004  Mr. Dalton, born in 1941 and resident in a sma...   
4  val_005  A parent, separated from their co-parent since...   
5  val_006  On 3 March 2012, homeowners Ms. L and her part...   
6  val_007  An heirship claims title to a vintage pocket c...   

                                      gold_citations  
0  Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....  
1  Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...  
2  Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...  
3  Art. 505 Abs. 1 ZGB;Art. 467 ZGB;Art. 469 Abs....  
4  Art. 133 Abs. 1 ZGB;Art. 133 Abs. 2 ZGB;Art. 2...  
5  Art. 1 Abs. 1 OR;Art. 18 Abs. 1 OR;Art. 363 OR...  
6  Art. 98 Abs. 2 IPRG;Art. 100 Abs. 1 IPRG;Art. ...  , 

i = 7 
  query_id            

In [ ]:
def regex_retrieve(self) -> list[int]:
    """
    Use extract_and_clean_citations() on the raw query to get citation strings.
    Then look those strings up in corpus['citation'] to get row indices.
    Returns a list of matched row indices (treated as highest-confidence hits).
    If no citations are found in the query, returns an empty list.
    """
    # Call your existing function directly
    citation_strings = extract_and_clean_citations(self.query_text)
 
    if not citation_strings:
            return []
 
    # Match extracted citation strings against the corpus citation column
    # Use str.contains for partial matching (handles "Art. 641 ZGB" matching
    # "Art. 641 Abs. 1 ZGB" in corpus, for example)
    matched_indices = []
    for cit in citation_strings:
        mask = self.corpus["citation"].str.contains(
            cit, case=False, na=False, regex=False
        )
        matched_indices.extend(self.corpus.index[mask].tolist())

    # Deduplicate while preserving order (first match = highest confidence)
    seen = set()
    unique_indices = []
    for idx in matched_indices:
        if idx not in seen:
            seen.add(idx)
            unique_indices.append(idx)

    print(f"Regex found {len(citation_strings)} citation(s) in query "
            f"→ matched {len(unique_indices)} corpus rows.")
    return unique_indices

In [ ]:
def retrieve(self, top_k: int = 10) -> list[dict]:
        """
        Run all three retrievers, fuse with RRF, return top_k results.
 
        Returns
        -------
        List of dicts with keys:
            citation    : str   — from corpus['citation']
            text        : str   — chunk text
            rrf_score   : float — fused score (higher = more relevant)
            rank        : int   — 1 = best
            bm25_rank   : int or None
            faiss_rank  : int or None
            regex_match : bool  — True if this row was found by regex
        """
        try:
            print(f"\nHybrid retrieval (BM25 + FAISS + Regex)")
            print(f"Query: '{self.query_text[:80]}...'")
 
            # ── Step 1: BM25 ──────────────────────────────────────────────────
            # BM25.Best_Match() → np.ndarray of top corpus row-indices
            bm25_indices = self.bm25_retriever.Best_Match()
            bm25_ranked  = bm25_indices.tolist()
 
            # ── Step 2: FAISS ─────────────────────────────────────────────────
            # Faiss.Faiss_retriever() → (scores, indices), shape (n_queries, top_k)
            # Take row 0 — single query
            _, dense_indices_full = self.faiss_retriever.Faiss_retriever()
            faiss_ranked = dense_indices_full[0].tolist()
 
            # ── Step 3: Regex ─────────────────────────────────────────────────
            # extract_and_clean_citations() → citation strings → corpus row indices
            regex_ranked = self._regex_retrieve()
 
            # ── Step 4: Fuse all three with RRF ──────────────────────────────
            # Build the list of ranked lists — only include regex if it found hits.
            # Regex hits go first in the list: they are treated as rank-1 evidence.
            ranked_lists = []
            if regex_ranked:
                ranked_lists.append(regex_ranked)   # highest confidence, goes first
            ranked_lists.append(bm25_ranked)
            ranked_lists.append(faiss_ranked)
 
            rrf_scores = reciprocal_rank_fusion(ranked_lists, k=self.rrf_k)
 
            # ── Step 5: Sort by fused score ───────────────────────────────────
            sorted_candidates = sorted(
                rrf_scores.items(),
                key=lambda x: x[1],
                reverse=True
            )
 
            # ── Step 6: Build rank lookup for diagnostics ─────────────────────
            bm25_rank_lookup  = {idx: r + 1 for r, idx in enumerate(bm25_ranked)}
            faiss_rank_lookup = {idx: r + 1 for r, idx in enumerate(faiss_ranked)}
            regex_match_set   = set(regex_ranked)
 
            # ── Step 7: Build result list ─────────────────────────────────────
            has_citation = "citation" in self.corpus.columns
            results = []
 
            for final_rank, (corpus_idx, score) in enumerate(
                sorted_candidates[:top_k], start=1
            ):
                row = self.corpus.iloc[corpus_idx]
                results.append({
                    "citation":    row["citation"] if has_citation else f"idx_{corpus_idx}",
                    "text":        str(row.get("text", "")),
                    "rrf_score":   round(score, 6),
                    "rank":        final_rank,
                    "bm25_rank":   bm25_rank_lookup.get(corpus_idx),
                    "faiss_rank":  faiss_rank_lookup.get(corpus_idx),
                    "regex_match": corpus_idx in regex_match_set,
                })
 
            regex_hits = sum(1 for r in results if r["regex_match"])
            print(f"Retrieved {len(results)} results | "
                  f"{regex_hits} confirmed by regex | top_k={top_k}")
            return results
 
        except Exception as e:
            raise Agentic_Exception(e, sys) from e
 
def retrieve_as_dataframe(self, top_k: int = 10) -> pd.DataFrame:
    """Same as retrieve() but returns a pandas DataFrame."""
    return pd.DataFrame(self.retrieve(top_k=top_k))
 

In [9]:
val.index[2]

2

In [ ]:
# ── Main retrieve ─────────────────────────────────────────────────────────

def retrieve(self, top_k: int = 10) -> list[dict]:
    """
    Run all three retrievers, fuse with RRF, return top_k results.

    Returns
    -------
    List of dicts with keys:
        citation    : str   — from corpus['citation']
        text        : str   — chunk text
        rrf_score   : float — fused score (higher = more relevant)
        rank        : int   — 1 = best
        bm25_rank   : int or None
        faiss_rank  : int or None
        regex_match : bool  — True if this row was found by regex
    """
    try:
        print(f"\nHybrid retrieval (BM25 + FAISS + Regex)")
        print(f"Query: '{self.query_text[:80]}...'")

        # ── Step 1: BM25 ──────────────────────────────────────────────────
        # BM25.Best_Match() → np.ndarray of top corpus row-indices
        bm25_indices = self.bm25_retriever.Best_Match()
        bm25_ranked  = bm25_indices.tolist()

        # ── Step 2: FAISS ─────────────────────────────────────────────────
        # Faiss.Faiss_retriever() → (scores, indices), shape (n_queries, top_k)
        # Take row 0 — single query
        _, dense_indices_full = self.faiss_retriever.Faiss_retriever()
        faiss_ranked = dense_indices_full[0].tolist()

        # ── Step 3: Regex ─────────────────────────────────────────────────
        # extract_and_clean_citations() → citation strings → corpus row indices
        regex_ranked = self._regex_retrieve()

        # ── Step 4: Fuse all three with RRF ──────────────────────────────
        # Build the list of ranked lists — only include regex if it found hits.
        # Regex hits go first in the list: they are treated as rank-1 evidence.
        ranked_lists = []
        if regex_ranked:
            ranked_lists.append(regex_ranked)   # highest confidence, goes first
        ranked_lists.append(bm25_ranked)
        ranked_lists.append(faiss_ranked)

        rrf_scores = reciprocal_rank_fusion(ranked_lists, k=self.rrf_k)

        # ── Step 5: Sort by fused score ───────────────────────────────────
        sorted_candidates = sorted(
            rrf_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        # ── Step 6: Build rank lookup for diagnostics ─────────────────────
        bm25_rank_lookup  = {idx: r + 1 for r, idx in enumerate(bm25_ranked)}
        faiss_rank_lookup = {idx: r + 1 for r, idx in enumerate(faiss_ranked)}
        regex_match_set   = set(regex_ranked)

        # ── Step 7: Build result list ─────────────────────────────────────
        has_citation = "citation" in self.corpus.columns
        results = []

        for final_rank, (corpus_idx, score) in enumerate(
            sorted_candidates[:top_k], start=1
        ):
            row = self.corpus.iloc[corpus_idx]
            results.append({
                "citation":    row["citation"] if has_citation else f"idx_{corpus_idx}",
                "text":        str(row.get("text", "")),
                "rrf_score":   round(score, 6),
                "rank":        final_rank,
                "bm25_rank":   bm25_rank_lookup.get(corpus_idx),
                "faiss_rank":  faiss_rank_lookup.get(corpus_idx),
                "regex_match": corpus_idx in regex_match_set,
            })

        regex_hits = sum(1 for r in results if r["regex_match"])
        print(f"Retrieved {len(results)} results | "
                f"{regex_hits} confirmed by regex | top_k={top_k}")
        return results

    except Exception as e:
        raise Agentic_Exception(e, sys) from e

def retrieve_as_dataframe(self, top_k: int = 10) -> pd.DataFrame:
    """Same as retrieve() but returns a pandas DataFrame."""
    return pd.DataFrame(self.retrieve(top_k=top_k))


In [1]:
my_dict = {
    1: 100,
    2: 200,
    3: 300,
    4: 400,
    5: 500,
    6: 600,
    7: 700,
    8: 800,
    9: 900,
    10: 1000
}

In [4]:
sort = sorted(my_dict.items(), key=lambda x: x[1])

In [5]:
sort

[(1, 100),
 (2, 200),
 (3, 300),
 (4, 400),
 (5, 500),
 (6, 600),
 (7, 700),
 (8, 800),
 (9, 900),
 (10, 1000)]

In [4]:
law.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [6]:
law.reset_index(drop=True)

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
...,...,...,...
175928,Art. 9 Abs. 4 ZWV,4 Das Grundbuchamt versieht auf Antrag des Eig...,Zweitwohnungsverordnung vom 4. Dezember 2015 (...
175929,Art. 10 Abs. 1 ZWV,1 Das ARE ist im Bereich des Zweitwohnungswese...,Zweitwohnungsverordnung vom 4. Dezember 2015 (...
175930,Art. 10 Abs. 2 ZWV,2 Die Baubewilligungsbehörden eröffnen dem ARE...,Zweitwohnungsverordnung vom 4. Dezember 2015 (...
175931,Art. 12 ZWV,Die nachstehenden Erlasse werden wie folgt geä...,Zweitwohnungsverordnung vom 4. Dezember 2015 (...


In [ ]:
citation = row.get('citation',f'law_idx_{idx}')

In [11]:
law.iloc[5]

citation                                    Art. 4 Abs. 2 112
text        2 Sie übernimmt auch die Verpflichtung, die er...
title       Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
Name: 5, dtype: object

In [12]:
law.iloc[5].get('citation')

'Art. 4 Abs. 2 112'